
# 📕 C04 — Reconstruction of Non-Cartesian MRI Data

> **Module 4 · Day 23 of the ROADMAP** — *Computational MRI, Lecture 4 (FAU Erlangen-Nürnberg / NYU)*
>
> *Real MRI doesn't always sample k-space on a nice rectangular grid. Here's how we turn radial, spiral, and EPI data into images.*

So far (Modules 1–3) every image came from **Cartesian** k-space: sample on a rectangular grid, press `ifft2`, done. But many of the most useful sequences — radial, spiral, EPI — trace **curved or rotated paths** through k-space. For those, the inverse FFT alone **does not work**, because the FFT assumes equally-spaced samples on a grid.

This notebook builds the classical non-Cartesian reconstruction toolkit **from scratch**, then shows the modern shortcut (NUFFT). It is split into two halves:

* **Part 1 — Training.** Concept-by-concept, following the lecture: trajectories → the CT connection (Radon / Fourier-slice / filtered backprojection) → gridding → density compensation → oversampling → deapodization → Kaiser–Bessel kernels → NUFFT.
* **Part 2 — Lab 4 exercises, *with solutions*.** The six lab tasks, each with a worked answer, plots, and discussion.

### What you'll be able to do by the end
- Generate a radial golden-angle k-space trajectory and reason about its Nyquist requirement
- Reconstruct non-Cartesian data by **gridding** (convolve → resample → FFT) and explain every step
- Apply **density compensation**, **oversampling**, and **deapodization**, and say *why* each is needed
- Use a research-grade **NUFFT** toolbox and compare it against your hand-written gridding

### A note on data
The lab uses a file `radial_data.mat`. Since that file isn't shipped with the repo, this notebook **synthesises an equivalent golden-angle radial dataset** from a Shepp–Logan phantom (so everything is runnable end-to-end). The very last cell shows exactly how to swap in your own `radial_data.mat` instead — the rest of the code is unchanged.


---
## Part 0 — Setup

Core scientific stack plus `sigpy` (used only for the forward simulation and for the NUFFT exercise). Install once if needed:

```bash
pip install numpy scipy scikit-image matplotlib sigpy
```

In [ ]:
import numpy as np
import scipy.fft as fft
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['image.cmap']     = 'gray'

# --- centered 2D (inverse) FFT helpers -----------------------------------
# MR k-space has its origin in the *centre* of the array, so we fftshift
# before and after every transform.
def fft2c(x):
    return fft.fftshift(fft.fft2(fft.ifftshift(x)))

def ifft2c(x):
    return fft.fftshift(fft.ifft2(fft.ifftshift(x)))

def show(img, title='', ax=None, log=False):
    if ax is None:
        _, ax = plt.subplots()
    m = np.abs(img)
    if log:
        m = np.log(m + 1e-3)
    ax.imshow(m); ax.set_title(title); ax.axis('off')
    return ax

def nrmse(a, b):
    a, b = np.abs(a), np.abs(b)
    a, b = a / a.max(), b / b.max()
    return np.sqrt(np.mean((a - b) ** 2))

In [ ]:
# Our ground-truth object: a Shepp-Logan phantom (a classic synthetic head/brain phantom)
N = 256
phantom = resize(shepp_logan_phantom(), (N, N), mode='reflect', anti_aliasing=True).astype(np.float64)

show(phantom, f'Shepp-Logan phantom ({N}x{N})')
plt.show()


---
# Part 1 — Training

## 1.1 Non-Cartesian trajectories: radial, EPI, spiral

A **trajectory** is the path the acquisition takes through k-space as the readout gradients play out. Cartesian sampling fills k-space line-by-line on a grid. The three classic non-Cartesian patterns:

| Trajectory | Path | Notes |
|---|---|---|
| **Radial** | straight spokes through the centre | the original Lauterbur 1973 scheme; CT-like |
| **EPI** | a fast zig-zag raster | covers k-space in one (or few) shots |
| **Spiral** | an outward spiral | very efficient gradient usage |

**Why bother?** Non-Cartesian readouts are typically **fast**, **motion-robust**, and offer **self-navigation** (the centre of k-space is re-measured every shot). The price:

* sensitive to **hardware imperfections** — off-resonance, gradient non-linearity, eddy currents;
* **numerically harder** to reconstruct — which is the entire subject of this notebook.

Let's just draw the three patterns so the shapes are concrete.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

# Radial: spokes through the origin
n_spokes_demo = 16
for a in np.linspace(0, np.pi, n_spokes_demo, endpoint=False):
    r = np.linspace(-0.5, 0.5, 2)
    ax[0].plot(r*np.cos(a), r*np.sin(a), 'C0')
ax[0].set_title('Radial')

# EPI: zig-zag raster
ky = np.repeat(np.linspace(-0.5, 0.5, 9), 2)
kx = np.tile([-0.5, 0.5, 0.5, -0.5], 5)[:len(ky)]
ax[1].plot(kx, ky, 'C1'); ax[1].set_title('EPI')

# Spiral
t = np.linspace(0, 1, 1000)
turns = 6
ax[2].plot(0.5*t*np.cos(2*np.pi*turns*t), 0.5*t*np.sin(2*np.pi*turns*t), 'C2')
ax[2].set_title('Spiral')

for a in ax:
    a.set_aspect('equal'); a.set_xlabel('$k_x$'); a.set_ylabel('$k_y$')
    a.set_xlim(-0.6, 0.6); a.set_ylim(-0.6, 0.6)
plt.tight_layout(); plt.show()


## 1.2 Why the direct FFT fails

The MR signal equation is the same as always — k-space is the Fourier transform of the object:

$$ s(k_x,k_y) = \iint m(x,y)\, e^{-i 2\pi (k_x x + k_y y)}\, dx\, dy $$

The `ifft2` you used for Cartesian data is the **inverse Discrete Fourier Transform**, and the DFT is only defined for samples sitting on a **uniform rectangular grid**. Radial/spiral/EPI samples land *between* grid points, so there's no array to hand to `ifft2`.

Two ways out:

1. **DFT according to the trajectory** — evaluate $m(x,y)=\sum_j s(k_j)\,e^{+i2\pi(k_{x,j}x+k_{y,j}y)}$ directly at every sample. Exact, but $O(N^2 \cdot N_\text{samp})$ — painfully **slow**.
2. **Gridding** — interpolate the scattered samples onto a Cartesian grid, then use the **fast** FFT. This is what everyone actually does, and it's the heart of this notebook.

For the special case of **radial** data there's also a third, historically-first route inherited straight from CT: **filtered backprojection**. We look at that next, because it explains *why density compensation exists*.


## 1.3 The CT connection — Radon transform, Fourier-slice theorem, (filtered) backprojection

Lauterbur's original 1973 MRI experiment used **radial** sampling and reconstructed with **filtered backprojection**, exactly like a CT scanner. The vocabulary transfers directly.

**Radon transform / projection.** A projection $P(s,\phi)$ integrates the object along parallel lines at angle $\phi$:

$$ P(s,\phi)=\iint I(x,y)\,\delta(x\cos\phi + y\sin\phi - s)\,dx\,dy $$

Stacking projections for all angles gives the **sinogram** (a point becomes a sine wave — hence the name).

**Fourier-slice (central-slice) theorem.** The 1-D Fourier transform of the projection at angle $\phi$ is exactly the **radial line of k-space at angle $\phi$**. So a radial MRI spoke *is* a projection's Fourier transform — you can hop between sinogram-space and radial k-space with a 1-D FFT.

**Backprojection.** To invert, smear each projection back across the image along its angle and sum over all angles:

$$ b(x,y)=\sum_{m} P\big(x\cos\phi_m + y\sin\phi_m,\ \phi_m\big) $$

Let's see all three on the phantom.

In [ ]:
from skimage.transform import radon, iradon

theta = np.linspace(0., 180., max(phantom.shape), endpoint=False)
sino  = radon(phantom, theta=theta)                       # projections (sinogram)
bp    = iradon(sino, theta=theta, filter_name=None)       # plain backprojection
fbp   = iradon(sino, theta=theta, filter_name='ramp')     # filtered backprojection

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
show(phantom, 'object', ax[0])
ax[1].imshow(sino, aspect='auto', extent=(0,180,sino.shape[0],0))
ax[1].set_title('Radon space (sinogram)'); ax[1].set_xlabel('angle $\\phi$'); ax[1].set_ylabel('s')
show(bp,  'backprojection (blurry!)', ax[2])
show(fbp, 'filtered backprojection',  ax[3])
plt.tight_layout(); plt.show()


**What went wrong with plain backprojection?** It's badly **blurred** — the low frequencies are massively over-emphasised while high-frequency detail is lost. The cause is purely geometric: radial spokes all pass through the **centre** of k-space, so the centre is measured many times while the periphery is sparse. This is **variable-density sampling** — the centre is acquired $\approx N$ times more than the edge.

**The fix — a ramp ("rho") filter.** Multiply k-space by $\rho(k)\propto|k|$ (rising from $\approx 1/N$ at the centre to $1$ at the edge). This *down-weights* the over-sampled centre and *boosts* the under-sampled edge, exactly compensating the radial density. Adding the ramp turns blurry backprojection into the sharp **filtered backprojection** above.

That ramp filter is the CT-flavoured name for what MRI calls **density compensation** — we'll meet it again in §1.5. Filtered backprojection works *only* for radial ("radiate") data though; **gridding works for any trajectory**, so that's where we go next.


## 1.4 Gridding — the general recipe

> **convolve → resample → inverse FFT**

Interpolate (convolve) the scattered non-Cartesian samples onto a Cartesian grid with a small **kernel**, then FFT. Mathematically, sampling is a sum of deltas at the trajectory locations,

$$ S(k_x,k_y)=\sum_i \delta(k_x-k_{x,i},\,k_y-k_{y,i}), $$

the measured data is $M(k_x,k_y)\,S(k_x,k_y)$, and gridding convolves that with a kernel $C$ and resamples on the Cartesian comb $\mathrm{III}$:

$$ \hat M(k_x,k_y)=\Big[\big(M\,S\big)\ast C\Big]\times \mathrm{III}\!\left(\tfrac{k_x}{\Delta k_x},\tfrac{k_y}{\Delta k_y}\right). $$

By the convolution theorem, **convolution in k-space = multiplication in image space**, so after the inverse FFT the recovered image is the true image **multiplied** by $c(x,y)=\mathcal F^{-1}\{C\}$ and **replicated** by the FOV comb:

$$ \hat m(x,y)=\Big[\big(m\ast s\big)\,c\Big]\ast\mathrm{III}\!\left(\tfrac{x}{FOV_x},\tfrac{y}{FOV_y}\right). $$

Each operation leaves a fingerprint we'll have to clean up:

| k-space operation | image-space effect | cure |
|---|---|---|
| sampling density not uniform | blurring / shading | **density compensation** (§1.5) |
| convolution with finite kernel | **apodization** (intensity roll-off) + side-lobes | **deapodization** (§1.7) + oversampling (§1.6) |
| resampling onto finite grid | **replication** (aliasing) | oversample the grid (§1.6) |

Here is a minimal **triangular** (bilinear) gridding kernel of width 2 — this is exactly the `grid1` you'll use in the lab.

In [ ]:
def grid1(d, k, n):
    '''Grid 2D non-Cartesian k-space data onto a Cartesian grid.

    Triangular (bilinear) kernel of width 2: each sample is spread onto the
    four nearest grid points with weights (1-|dx|)(1-|dy|).

    Parameters
    ----------
    d : complex array         non-Cartesian k-space samples (any shape)
    k : complex array          trajectory, real=kx imag=ky, scaled -0.5..0.5
    n : int                    output grid size
    Returns
    -------
    (n, n) complex Cartesian k-space grid
    '''
    d  = np.asarray(d).ravel()
    k  = np.asarray(k).ravel()
    gx = (np.real(k) + 0.5) * n          # map -0.5..0.5  ->  0..n
    gy = (np.imag(k) + 0.5) * n
    out = np.zeros((n, n), dtype=np.complex128)
    fx, fy = np.floor(gx), np.floor(gy)
    for ox in (0, 1):                    # the 4 nearest grid points
        for oy in (0, 1):
            ix = (fx + ox).astype(int)
            iy = (fy + oy).astype(int)
            w  = (1 - np.abs(gx - ix)) * (1 - np.abs(gy - iy))
            ok = (ix >= 0) & (ix < n) & (iy >= 0) & (iy < n) & (w > 0)
            np.add.at(out, (iy[ok], ix[ok]), (d * w)[ok])
    return out


### A radial dataset to play with

We build a golden-angle radial trajectory and *simulate* the samples a scanner would measure, using a forward NUFFT on the phantom. The result, `data`, plays the role of `radial_data.mat`.

* `n_read` samples per spoke (the readout direction)
* `n_spokes` spokes, each rotated by the **golden angle** $111.246117975^\circ$
* trajectory `k_norm` is complex (`real`=$k_x$, `imag`=$k_y$), scaled to $[-0.5, 0.5]$ — exactly what `grid1` expects

In [ ]:
import sigpy

n_read   = 2 * N                          # oversampled readout
n_spokes = 402                            # ~ Nyquist for N=256 (see Exercise 1c)
GA = np.deg2rad(111.246117975)            # 2D golden angle

kr  = (np.arange(n_read) - n_read / 2) / n_read       # -0.5 .. ~0.5 along a spoke
ang = np.arange(n_spokes) * GA                        # cumulative golden-angle rotation
KX  = np.outer(kr, np.cos(ang))                       # (n_read, n_spokes): col = one spoke
KY  = np.outer(kr, np.sin(ang))
k_norm = KX + 1j * KY                                 # trajectory scaled -0.5..0.5

# forward simulate the acquired samples (coords for sigpy are in matrix units -N/2..N/2)
coord = np.stack([KX * N, KY * N], axis=-1)           # (n_read, n_spokes, 2)
data  = sigpy.nufft(phantom, coord)                   # <- this is our "radial_data"
print('data (radial k-space):', data.shape, data.dtype)


Now reconstruct it the naive way — grid with the triangular kernel and inverse-FFT, **with no other correction** — to see the raw gridding fingerprint.

In [ ]:
g_raw   = grid1(data, k_norm, N)
img_raw = ifft2c(g_raw)

fig, ax = plt.subplots(1, 2)
show(g_raw,   'gridded k-space (log)', ax[0], log=True)
show(img_raw, 'naive gridding recon',  ax[1])
plt.tight_layout(); plt.show()


That bright central blob is the **density** problem from §1.3 showing up in gridding too: the radial centre is hugely over-weighted. Time to fix it.

## 1.5 Density compensation

Non-Cartesian trajectories sample k-space with **variable density**. For radial, the central point is hit once per spoke — $N$ times total — while the periphery is sparse. Before gridding we **pre-compensate**: weight each sample by the inverse of its local sampling density $\rho$,

$$ \hat M = \Big[\big(\tfrac{M}{\rho}\,S\big)\ast C\Big]\times\mathrm{III}. $$

For radial the density is known analytically — it falls off as $1/|k|$ — so the weight is the **ramp** $\rho^{-1}\propto|k|$ (rising from $\approx 1/N$ at the centre to $1$ at the edge), exactly the CT ramp filter from §1.3. For arbitrary trajectories you compute the per-sample area numerically (e.g. a **Voronoi** diagram).

In [ ]:
def ramp_dcf(kr, n_spokes):
    '''Radial density-compensation weights: |k| ramp, one row per readout sample.'''
    w = np.abs(kr)[:, None] * np.ones((1, n_spokes))
    w = w / w.max()
    w[len(kr) // 2, :] = 1.0 / n_spokes          # small floor at the DC sample
    return w

w = ramp_dcf(kr, n_spokes)
img_dcf = ifft2c(grid1(data * w, k_norm, N))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
show(img_raw, 'no compensation', ax[0])
show(img_dcf, 'with ramp / density comp', ax[1])
ax[2].plot(np.abs(kr), w[:, 0]); ax[2].set_title('ramp weight $\\rho^{-1}\\propto|k|$')
ax[2].set_xlabel('|k|'); ax[2].set_ylabel('weight')
plt.tight_layout(); plt.show()
print('NRMSE  no-DCF = %.3f   |   with-DCF = %.3f' % (nrmse(img_raw, phantom), nrmse(img_dcf, phantom)))


The phantom snaps into focus. Residual faint **streaks** are golden-angle radial under-sampling, and the lingering softness is the crude triangular kernel — addressed by oversampling, deapodization, and a better kernel below.

## 1.6 Oversampling the Cartesian grid

The finite kernel causes two image-space problems: **replication / aliasing** (the $\mathrm{III}$ comb folds neighbouring copies into the FOV) and **apodization** (intensity roll-off toward the edges). The cheap, brutal fix for both: **grid onto a larger grid** (e.g. $2N\times 2N$), inverse-FFT, then **crop** the central $N\times N$ in image space.

Why it helps: a bigger grid means a finer image-domain sampling, so the aliased copies are pushed *outside* the FOV, and the steep part of the apodization roll-off lands *outside* the cropped region — leaving only mild residual shading.

In [ ]:
def decimate2d(img, factor):
    '''Crop the central 1/factor region of an oversampled image (image-domain decimation).'''
    n  = img.shape[0]
    m  = n // factor
    s  = (n - m) // 2
    return img[s:s + m, s:s + m]

os = 2
img_os_full = ifft2c(grid1(data * w, k_norm, N * os))
img_os      = decimate2d(img_os_full, os)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
show(img_os_full, f'{os}X grid (note corner aliasing)', ax[0])
show(img_os,      'cropped to FOV', ax[1])
show(img_dcf,     '1X grid (for comparison)', ax[2])
plt.tight_layout(); plt.show()


## 1.7 Deapodization

Convolving with the kernel $C(k)$ in k-space multiplies the image by $c(x)=\mathcal F^{-1}\{C\}$ — the **apodization** function, bright in the centre and dimming toward the edges. We undo it by **dividing the reconstructed image by $c(x)$** (with a tiny constant to avoid dividing by zero):

$$ \hat m(x,y)=\frac{\big[(m\ast s)\,c\big]\ast\mathrm{III}}{c(x,y)+a}. $$

For the triangular (width-2) kernel, $C$ is a triangle and its transform is $c(x)=\operatorname{sinc}^2(x/n)$. (For a general kernel you'd instead grid a single delta and inverse-FFT to read $c(x)$ off numerically — see Exercise 5.)

In [ ]:
def sinc2_apod(n):
    x = np.arange(n) - n // 2
    a1 = np.sinc(x / n) ** 2                    # transform of width-2 triangular kernel
    return np.outer(a1, a1)

a_full   = sinc2_apod(N * os)
img_deap = decimate2d(img_os_full / (a_full + 1e-3), os)

# profile through the middle row
row = N // 2
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(img_os,   'before deapodization', ax[0])
show(img_deap, 'after deapodization',  ax[1])
ax[2].plot(np.abs(img_os)[row],   label='before')
ax[2].plot(np.abs(img_deap)[row], label='after')
ax[2].set_title('central profile (edges lifted)'); ax[2].legend()
plt.tight_layout(); plt.show()


With $2\times$ oversampling the apodization across the FOV is gentle (centre $1.0 \to$ edge $\approx 0.81$), so deapodization is a **small** correction here — a point we return to in Exercise 3d. Without oversampling, or with a wider kernel, it matters much more.

## 1.8 Better kernels — Kaiser–Bessel

The triangular kernel is fast but crude (broad point-spread, strong side-lobes). The **ideal** kernel is an infinite sinc (its transform is a perfect rect = no apodization, no aliasing) — but infinite support is impractical. So we use a **windowed sinc**; by consensus the best practical choice is the **Kaiser–Bessel** kernel:

$$ C(k)=\frac{1}{W}\,I_0\!\Big(b\sqrt{1-(2k/W)^2}\Big)\,\operatorname{rect}\!\Big(\tfrac{2k}{W}\Big), \qquad c(x)=\frac{\sin\sqrt{\pi^2 W^2 x^2 - b^2}}{\sqrt{\pi^2 W^2 x^2 - b^2}} $$

where $I_0$ is the zeroth-order modified Bessel function, $W$ the kernel width, $b$ a shape parameter. It concentrates energy in the main lobe far better than a triangle for the same width.

In [ ]:
from scipy.special import i0

W, b = 4.0, 6.0
kk  = np.linspace(-W/2, W/2, 400)
kb  = i0(b * np.sqrt(np.clip(1 - (2*kk/W)**2, 0, None))) / W
kkt = np.linspace(-1, 1, 400)
tri = np.clip(1 - np.abs(kkt), 0, None)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(kk,  kb / kb.max(), label='Kaiser-Bessel (W=4)')
ax[0].plot(kkt, tri,           label='triangular (W=2)')
ax[0].set_title('gridding kernels in k-space'); ax[0].set_xlabel('k (grid units)'); ax[0].legend()

# image-domain response = |FT of the kernel|; lower side-lobes => less aliasing energy
def apod(kernel, pad=4096):
    g = np.zeros(pad); c = pad // 2
    half = len(kernel) // 2
    g[c-half:c-half+len(kernel)] = kernel
    A = np.abs(fft.fftshift(fft.fft(fft.ifftshift(g))))
    return A / A.max()

# sample both kernels on the SAME fine k-grid so the comparison is fair
kfine = np.linspace(-2, 2, 4001)
kb_f  = np.where(np.abs(kfine) <= W/2,
                 i0(b * np.sqrt(np.clip(1 - (2*kfine/W)**2, 0, None))) / W, 0.0)
tri_f = np.clip(1 - np.abs(kfine), 0, None)

Akb, Atri = apod(kb_f), apod(tri_f)
xx = (np.arange(len(Akb)) - len(Akb)//2) / 50.0      # normalised image position

m = np.abs(xx) < 4
ax[1].semilogy(xx[m], Atri[m], label='triangular (W=2)')
ax[1].semilogy(xx[m], Akb[m],  label='Kaiser-Bessel (W=4)')
ax[1].set_ylim(1e-5, 1.5)
ax[1].set_title('image-domain response (log) — lower side-lobes = less aliasing')
ax[1].set_xlabel('normalised image position'); ax[1].legend()
plt.tight_layout(); plt.show()


## 1.9 NUFFT — the modern shortcut

The **Non-Uniform FFT** is a generalised, highly-optimised gridding: same idea (kernel-interpolate + FFT, with a good Kaiser–Bessel kernel, oversampling and deapodization all built in), but packaged as fast **forward and adjoint** operators you can drop straight into iterative reconstructions (Module 4 later, and the AI recon of Module 5). Well-known implementations: `sigpy`, `torchkbnufft`, `BART`, `gpuNUFFT`, Fessler's IRT.

One line reconstructs our radial data (the adjoint = "grid + FFT"):

In [ ]:
img_nufft = sigpy.nufft_adjoint(data * w, coord, oshape=(N, N))   # with density compensation

fig, ax = plt.subplots(1, 2)
show(img_dcf,   'hand-written gridding (triangular)', ax[0])
show(img_nufft, 'sigpy NUFFT (Kaiser-Bessel)',        ax[1])
plt.tight_layout(); plt.show()
print('NRMSE  triangular = %.3f   |   NUFFT = %.3f' % (nrmse(img_dcf, phantom), nrmse(img_nufft, phantom)))


The NUFFT's better kernel resolves the small dots and suppresses the streak/blur the triangular kernel left behind — the whole point of §1.8.

---
# Part 2 — Lab 4 exercises (with solutions)

> **Computational MR Imaging — Laboratory 4: Reconstruction of non-Cartesian k-space data**
>
> *Learning objectives:* reconstruct non-Cartesian MRI data using gridding and a NUFFT toolbox; apply density compensation, oversampling and deapodization; learn to use the NUFFT toolbox.

Everything below reuses the `data`, `k_norm`, `coord`, `kr`, `grid1`, `ifft2c` defined in Part 1. If you have the real `radial_data.mat`, load it as shown in the **Appendix** and the same code runs unchanged.


## Exercise 1 — Radial sampling pattern

**(a)** Load `radial_data.mat` and plot the acquired k-space. Each *column* is the readout for one radial line, acquired with a golden-angle increment ($111.246117975^\circ$).
**(b)** Generate the sampling trajectory for the reconstruction (Figure 1 shows the first 10 spokes).
**(c)** If the Cartesian matrix size is $384\times384$, how many radial lines are needed for the Nyquist rate?

In [ ]:
# --- 1(a) the acquired k-space: rows = readout, columns = spokes ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(np.log(np.abs(data) + 1e-3), aspect='auto', cmap='gray')
ax[0].set_title('acquired radial k-space  (log|data|)')
ax[0].set_xlabel('spoke index'); ax[0].set_ylabel('readout sample')

# --- 1(b) the trajectory; first 10 spokes, first angle at pi/2 like Figure 1 ---
for s in range(10):
    ax[1].plot(np.real(k_norm[:, s]), np.imag(k_norm[:, s]))
ax[1].set_aspect('equal'); ax[1].set_title('trajectory: first 10 spokes')
ax[1].set_xlabel('$k_x$'); ax[1].set_ylabel('$k_y$')
plt.tight_layout(); plt.show()


**1(c) — Nyquist number of radial lines for a $384\times384$ matrix.**

Along each spoke the samples are Nyquist-spaced, $\Delta k = 1/FOV$. The spokes span a full diameter and are distributed over **180°**. At the edge of k-space ($k_\max = N/(2\,FOV)$) the gap *between adjacent spokes* must also be $\le 1/FOV$. The arc length swept at the edge over $\pi$ radians is $\pi\,k_\max$, so the number of spokes is

$$ N_\text{spokes}=\frac{\pi\,k_\max}{\Delta k}=\frac{\pi\,\frac{N}{2\,FOV}}{1/FOV}=\frac{\pi}{2}\,N. $$

For $N=384$:  $\dfrac{\pi}{2}\times 384 \approx \mathbf{604}$ **radial lines**. (A factor $\approx 1.57N$ — radial imaging needs *more* lines than Cartesian phase-encodes to fully sample the periphery, which is why radial is so often deliberately under-sampled.)

In [ ]:
N_mat = 384
print('Nyquist radial lines for %dx%d :  pi/2 * N = %.1f  ->  %d spokes'
      % (N_mat, N_mat, np.pi/2*N_mat, int(np.ceil(np.pi/2*N_mat))))


## Exercise 2 — Basic gridding reconstruction

**(a)** Reconstruct using the provided `grid1` (triangular kernel, width 2).
**(b)** Plot the gridded k-space and the reconstructed image.
**(c)** Comment on the artifacts. Can you guess what organ was imaged?

In [ ]:
g2   = grid1(data, k_norm, N)        # (a) no density compensation yet
img2 = ifft2c(g2)

fig, ax = plt.subplots(1, 2)          # (b)
show(g2,   'gridded k-space (log)', ax[0], log=True)
show(img2, 'gridding recon (no DCF)', ax[1])
plt.tight_layout(); plt.show()


**2(c) — Artifacts & object.**
The recon is dominated by a bright central **blob**: with no density compensation the heavily-oversampled centre of radial k-space swamps the periphery, so low frequencies are over-weighted and high-frequency detail (edges) is suppressed — the same blurring we saw with plain backprojection (§1.3). Fainter **radial streaks** from the golden-angle spokes are also present.

The object here is the **Shepp–Logan phantom**, the standard synthetic *head / brain* phantom (outer skull-like ellipse, two ventricle-like ellipses, small lesions). With the original lab data the same recipe applies; radial acquisitions in the course material are often **abdominal** (the gridding slide shows a liver), so on real data you'd expect to recognise an abdomen/liver once density compensation is applied in Exercise 3.


## Exercise 3 — Density compensation

**(a)** Define a ramp filter. **(b)** Apply it and reconstruct. **(c)** Plot gridded k-space and image.
**(d)** Do you need oversampling and deapodization on this dataset? Explain.

In [ ]:
# 3(a) ramp filter (radial density compensation): weight ~ |k|
def ramp_filter(kr, n_spokes):
    w = np.abs(kr)[:, None] * np.ones((1, n_spokes))
    w = w / w.max()
    w[len(kr) // 2, :] = 1.0 / n_spokes        # avoid nulling the DC sample
    return w

w3   = ramp_filter(kr, n_spokes)               # 3(b)
g3   = grid1(data * w3, k_norm, N)
img3 = ifft2c(g3)

fig, ax = plt.subplots(1, 2)                    # 3(c)
show(g3,   'density-compensated k-space (log)', ax[0], log=True)
show(img3, 'recon with ramp filter', ax[1])
plt.tight_layout(); plt.show()
print('NRMSE  before = %.3f   after density comp = %.3f' % (nrmse(img2, phantom), nrmse(img3, phantom)))


**3(d) — Do we need oversampling / deapodization here?**

*Mostly no, for this dataset* — but for understandable reasons, and it's worth doing them to see why:

* **Oversampling (anti-aliasing).** Needed only if aliased copies fall inside the FOV. Here the object (the phantom) comfortably fills the reconstructed FOV with empty background around it, and the readout is already $2\times$ oversampled, so wrap-around is minimal. Oversampling the *grid* still cleans up faint edge replication, so it's cheap insurance rather than a necessity.
* **Deapodization.** The triangular kernel is **narrow (width 2)**, so its image-domain apodization is a gentle $\operatorname{sinc}^2$ that only dips to $\approx 0.4$ at the very edge of a $1\times$ grid — and *much* less than that across the FOV once you oversample. So the intensity shading is small and deapodization is a minor cosmetic correction here.

In short: density compensation is **essential** (huge effect), while oversampling + deapodization are **second-order** for a narrow kernel + an object that fits the FOV. They become important with wider kernels, smaller relative FOV, or quantitative work where flat intensity matters. Exercises 4–5 implement them anyway.


## Exercise 4 — Oversampling

**(a)** Define `decimate2d()`. **(b)** Oversample the density-compensated data by a factor of 2, inverse-FFT, and decimate. **(c)** Plot the gridded k-space and both reconstructions (oversampled & decimated).

In [ ]:
# 4(a)
def decimate2d(img, factor):
    '''Crop the central 1/factor of an (oversampled) image -> image-domain decimation.'''
    n = img.shape[0]; m = n // factor; s = (n - m) // 2
    return img[s:s + m, s:s + m]

# 4(b)
osf          = 2
g4           = grid1(data * w3, k_norm, N * osf)     # oversample the density-comp data
img4_full    = ifft2c(g4)                            # inverse FFT
img4         = decimate2d(img4_full, osf)            # decimate

# 4(c)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(g4,        f'gridded k-space {osf}X (log)', ax[0], log=True)
show(img4_full, 'oversampled recon (2N)',        ax[1])
show(img4,      'decimated to FOV (N)',          ax[2])
plt.tight_layout(); plt.show()


## Exercise 5 — Deapodization

Compute the deapodization function in the image domain and apply it to the oversampled (×2) gridded image.
*Hint: think of gridding a delta function.*

$$ \hat m(x,y)=\frac{1}{c(x,y)+a}\Big\{\big[(m\ast s)\,c(x,y)\big]\ast\mathrm{III}\big(\tfrac{x}{FOV_x},\tfrac{y}{FOV_y}\big)\Big\} $$

We use the hint directly: **grid a single unit sample at k = 0 and inverse-FFT** — the result is the kernel's image-domain footprint $c(x,y)$ (the apodization). Then divide. We also overlay the analytic $\operatorname{sinc}^2$ as a cross-check.

In [ ]:
# 5 -- deapodization function by gridding a delta (numerical), valid for ANY kernel
delta_grid = grid1(np.array([1.0 + 0j]), np.array([0.0 + 0j]), N * osf)
c_num      = np.abs(ifft2c(delta_grid)); c_num /= c_num.max()

# analytic sinc^2 for the triangular (width-2) kernel, as a cross-check
c_ana = sinc2_apod(N * osf)

a = 1e-3
img5 = decimate2d(img4_full / (c_ana + a), osf)        # apply deapodization, then crop

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(img4, 'before deapodization', ax[0])
show(img5, 'after deapodization',  ax[1])
r = (N * osf) // 2
ax[2].plot(c_ana[r] / c_ana[r].max(), label='analytic sinc$^2$')
ax[2].set_title('apodization $c(x)$ (centre row)'); ax[2].set_xlabel('grid x'); ax[2].legend()
plt.tight_layout(); plt.show()


The correction is subtle for this narrow kernel + ×2 grid (consistent with the answer to 3d): the centre row of $c(x)$ stays near 1 across the cropped FOV, so dividing mainly lifts the extreme edges. With a Kaiser–Bessel kernel (wider) the same machinery does real work — which is exactly what a NUFFT toolbox handles for you in Exercise 6.


## Exercise 6 — NUFFT toolbox

**(a)** Reconstruct the radial dataset with a widely-used NUFFT toolbox — both **with** and **without** density compensation — and discuss.
**(b)** Compare the NUFFT reconstruction with the triangular-kernel gridding.

We use `sigpy` (in `requirements.txt`). Its `nufft_adjoint` is the gridding-style "adjoint" operator; with the ramp weights it becomes a full density-compensated reconstruction.

In [ ]:
# 6(a)
img_nufft_nodcf = sigpy.nufft_adjoint(data,      coord, oshape=(N, N))   # without DCF
img_nufft_dcf   = sigpy.nufft_adjoint(data * w3, coord, oshape=(N, N))   # with DCF

fig, ax = plt.subplots(1, 2)
show(img_nufft_nodcf, 'NUFFT without density comp', ax[0])
show(img_nufft_dcf,   'NUFFT with density comp',    ax[1])
plt.tight_layout(); plt.show()

In [ ]:
# 6(b) compare triangular gridding vs NUFFT (both density-compensated)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(phantom,       'ground truth',                 ax[0])
show(img3,          'triangular gridding + DCF',     ax[1])
show(img_nufft_dcf, 'NUFFT (Kaiser-Bessel) + DCF',  ax[2])
plt.tight_layout(); plt.show()

print('NRMSE  triangular gridding + DCF : %.3f' % nrmse(img3, phantom))
print('NRMSE  NUFFT + DCF               : %.3f' % nrmse(img_nufft_dcf, phantom))


**Discussion.**
* **Density compensation dominates.** Without it, *both* methods collapse to the bright central blob — the variable radial density must be undone whatever interpolation you use.
* **Kernel quality is the differentiator.** With DCF applied, the NUFFT's optimised Kaiser–Bessel kernel (plus its internal oversampling + deapodization) gives a markedly lower error and resolves the small features the width-2 triangular kernel blurs and surrounds with side-lobes. That gap is precisely the §1.8 lesson, now quantified.
* **Why use the toolbox.** Beyond image quality, the NUFFT gives you fast, differentiable **forward & adjoint** operators — the building blocks for the iterative and deep-learning reconstructions in the rest of Module 4 and Module 5.


---
## 🧠 Self-test questions

1. Why can't you reconstruct radial or spiral data by calling `ifft2` directly?
2. State the Fourier-slice theorem in one sentence, and explain how it links a radial spoke to a CT projection.
3. Plain backprojection is blurry. In k-space terms, *why* — and what filter fixes it?
4. What is the radial density-compensation weight as a function of $|k|$, and where does the $\propto|k|$ come from geometrically?
5. Gridding leaves three image-space fingerprints (blur/shading, side-lobes, replication). Match each to its k-space cause.
6. Why does oversampling the Cartesian grid reduce *both* aliasing and apodization?
7. What is deapodization, and how can you obtain the deapodization function by "gridding a delta"?
8. Why is the Kaiser–Bessel kernel preferred over a triangular one of the same width?
9. For a $256\times256$ matrix, roughly how many radial spokes give Nyquist sampling? (Show the formula.)
10. Name two things a NUFFT operator gives you that a one-shot gridding recon does not.


---
## Appendix — using the real `radial_data.mat`

Everything above runs on synthetic data so the notebook is self-contained. To use the lab's actual file, replace the data-generation cell in §1.4 with the loader below and **keep the rest unchanged** — the trajectory `k_norm`/`coord` are reconstructed from the golden-angle scheme, and `grid1`, the ramp filter, oversampling, deapodization and the NUFFT calls all operate on `data` exactly as written.

In [ ]:
# from scipy.io import loadmat
# mat   = loadmat('radial_data.mat')
# data  = mat['radial_data']          # adjust key to whatever the .mat actually contains
# n_read, n_spokes = data.shape       # rows = readout, columns = spokes
#
# # rebuild the matching golden-angle trajectory, scaled -0.5..0.5
# GA  = np.deg2rad(111.246117975)
# kr  = (np.arange(n_read) - n_read/2) / n_read
# ang = np.arange(n_spokes) * GA
# KX, KY = np.outer(kr, np.cos(ang)), np.outer(kr, np.sin(ang))
# k_norm = KX + 1j*KY
# coord  = np.stack([KX*N, KY*N], axis=-1)   # for sigpy (matrix units); pick N to taste
#
# # ...then run Exercises 1-6 unchanged.